In [ ]:
## Part 1: Tumbling Window Dashboard (Estimated: 20 minutes)
# Build a real-time dashboard that shows per-minute metrics using tumbling windows.

# Task: Multi-Metric Tumbling Windows

# windowed_dashboard.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, window, count, avg, sum as _sum,
    min as _min, max as _max, countDistinct, to_timestamp,
    when, lit, round as _round
)
from pyspark.sql.types import StructType, StringType, IntegerType

spark = SparkSession.builder \
    .appName("Windowed-Dashboard") \
    .config("spark.sql.shuffle.partitions", "6") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Schema and Kafka source (same as Lab 1)
EVENT_SCHEMA = StructType() \
    .add("event_id", StringType()) \
    .add("user_id", StringType()) \
    .add("action", StringType()) \
    .add("content_id", StringType()) \
    .add("show_id", StringType()) \
    .add("device", StringType()) \
    .add("country", StringType()) \
    .add("timestamp", StringType()) \
    .add("duration_seconds", IntegerType()) \
    .add("session_id", StringType())

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "streaming.user.interactions") \
    .option("startingOffsets", "latest") \
    .load()

events = raw \
    .select(from_json(col("value").cast("string"), EVENT_SCHEMA).alias("e")) \
    .select("e.*") \
    .withColumn("event_time", to_timestamp(col("timestamp"))) \
    .withColumn(
        "engagement_score",
        when(col("action") == "play", 1)
        .when(col("action") == "complete", 3)
        .when(col("action") == "like", 5)
        .when(col("action") == "share", 7)
        .when(col("action") == "skip", -1)
        .otherwise(0)
    )

# ============================================
# Dashboard Metric 1: Events Per Minute by Action
# ============================================
events_per_minute = events \
    .withWatermark("event_time", "5 minutes") \
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("action")
    ) \
    .agg(
        count("*").alias("event_count"),
        countDistinct("user_id").alias("unique_users"),
    )

# ============================================
# Dashboard Metric 2: Engagement Per 5 Minutes by Show
# ============================================
show_engagement = events \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("show_id")
    ) \
    .agg(
        count("*").alias("total_events"),
        countDistinct("user_id").alias("unique_viewers"),
        _sum("engagement_score").alias("total_engagement"),
        _round(avg("engagement_score"), 2).alias("avg_engagement"),
        count(when(col("action") == "play", 1)).alias("plays"),
        count(when(col("action") == "complete", 1)).alias("completions"),
        count(when(col("action") == "skip", 1)).alias("skips"),
    )

# ============================================
# Dashboard Metric 3: Country Activity Per 5 Minutes
# ============================================
country_activity = events \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("country")
    ) \
    .agg(
        count("*").alias("events"),
        countDistinct("user_id").alias("users"),
    )

# Start all queries
q1 = events_per_minute.writeStream \
    .outputMode("update") \
    .format("console") \
    .queryName("events_per_minute") \
    .trigger(processingTime="30 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/dash-epm/") \
    .start()

q2 = show_engagement.writeStream \
    .outputMode("update") \
    .format("console") \
    .queryName("show_engagement") \
    .trigger(processingTime="30 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/dash-show/") \
    .start()

q3 = country_activity.writeStream \
    .outputMode("update") \
    .format("console") \
    .queryName("country_activity") \
    .trigger(processingTime="30 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/dash-country/") \
    .start()

spark.streams.awaitAnyTermination()


In [ ]:
+---------------------+----------------+------------------+-----------------+------------------+
| Window              | Top Action     | Top Show         | Top Country     | Total Events     |
+---------------------+----------------+------------------+-----------------+------------------+
| 10:30:00 - 10:31:00 | play (2)       | show-x-ep-5 (2)  | US (1), JP (1)  | 2                |
| 10:31:00 - 10:32:00 | purchase (1)   | premium-plan (1) | DE (1)          | 1                |
| 10:32:00 - 10:33:00 | skip (1)       | show-y-ep-2 (1)  | JP (1)          | 1                |


In [ ]:
## Part 2: Sliding Window Trend Detection (Estimated: 20 minutes)
# Build a trend detection system using sliding windows.

# Task: Detect Engagement Spikes and Drops
# trend_detector.py
"""
Detect engagement trends using sliding windows.
Alert when engagement deviates significantly from baseline.
"""

# ============================================
# Rolling 10-minute engagement, sliding every 2 minutes
# ============================================
rolling_engagement = events \
    .withWatermark("event_time", "15 minutes") \
    .groupBy(
        window(col("event_time"), "10 minutes", "2 minutes")
    ) \
    .agg(
        count("*").alias("event_count"),
        countDistinct("user_id").alias("unique_users"),
        avg("engagement_score").alias("avg_engagement"),
        count(when(col("action") == "skip", 1)).alias("skip_count"),
        count(when(col("action") == "play", 1)).alias("play_count"),
    ) \
    .withColumn(
        "skip_rate",
        _round(col("skip_count") / col("event_count") * 100, 1)
    ) \
    .withColumn(
        "events_per_minute",
        _round(col("event_count") / lit(10), 1)  # 10-min window
    )

# Implement YOUR alerting logic:
# - Alert if skip_rate > 20%
# - Alert if events_per_minute drops below a threshold
# - Alert if avg_engagement drops below baseline

q_trend = rolling_engagement \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .queryName("trend_detection") \
    .trigger(processingTime="30 seconds") \
    .option("checkpointLocation", "/tmp/checkpoints/trend/") \
    .start()


In [ ]:
+---------------------+---------------------+---------------+---------------+-----------------+------------------+
| Window Start        | Window End          | Events/Min    | Skip Rate     | Avg Engagement  | Trend            |
+---------------------+---------------------+---------------+---------------+-----------------+------------------+
| 10:30:00            | 10:40:00            | 0.6           | 16.7%         | 1.3             | Normal           |
| 10:32:00            | 10:42:00            | 0.4           | 25.0%         | 0.8             | ⚠️ High Skip Rate |
| 10:34:00            | 10:44:00            | 0.3           | 33.3%         | 0.7             | ⚠️ High Skip Rate |
| 10:36:00            | 10:46:00            | 0.2           | 50.0%         | 0.5             | 🔴 Engagement Drop|
| 10:38:00            | 10:48:00            | 0.1           | 100.0%        | -1.0            | 🔴 Critical       |
+---------------------+---------------------+---------------+---------------+-----------------+------------------+

In [ ]:
# Production alert logic:
# Add alert columns
rolling_with_alerts = rolling_engagement \
    .withColumn(
        "alert_level",
        when(col("skip_rate") > 20, "HIGH")
        .when(col("avg_engagement") < 0.8, "MEDIUM")
        .otherwise("LOW")
    ) \
    .withColumn(
        "alert_message",
        when(col("skip_rate") > 20, concat(lit("⚠️ Skip rate: "), col("skip_rate"), lit("%")))
        .when(col("avg_engagement") < 0.8, concat(lit("⚠️ Low engagement: "), col("avg_engagement")))
        .otherwise(lit("Normal"))
    )

In [ ]:
## Part 3: Session Windows - Explanation
# session_analytics.py
from pyspark.sql.functions import session_window

user_sessions = events \
    .withWatermark("event_time", "30 minutes") \
    .groupBy(
        session_window(col("event_time"), "15 minutes"),  # Session window with 15-min gap
        col("user_id")
    ) \
    .agg(
        count("*").alias("events_in_session"),
        _min("event_time").alias("session_start"),
        _max("event_time").alias("session_end"),
        countDistinct("content_id").alias("unique_content_watched"),
        countDistinct("show_id").alias("unique_shows"),
        _sum("engagement_score").alias("session_engagement"),
        count(when(col("action") == "play", 1)).alias("plays"),
        count(when(col("action") == "complete", 1)).alias("completions"),
        count(when(col("action") == "skip", 1)).alias("skips"),
        count(when(col("action") == "like", 1)).alias("likes"),
    )


In [ ]:
+---------------------+---------------------+------------------+-----------------+----------------------+
| User                | Session Duration    | Events           | Completion Rate | Engagement/Min       |
+---------------------+---------------------+------------------+-----------------+----------------------+
| U-1234              | 5.0 min             | 2                | 0.0%            | 0.4                  |
| U-5678              | 1.0 min             | 1                | 100.0%          | 7.0                  |
| U-9012              | 1.0 min             | 1                | 0.0%            | -1.0                 |
| U-3456              | 1.0 min             | 1                | 0.0%            | 5.0                  |
| U-7890              | 1.0 min             | 1                | 0.0%            | 1.0                  |
+---------------------+---------------------+------------------+-----------------+----------------------+

In [ ]:
## Part 4: Multi-Window Comparison
# Task: Run All Three Window Types and Compare
# comparison.py
"""
Compare tumbling, sliding, and session windows on the same data.
"""

# Metric: total engagement

# Tumbling: 5-min buckets
tumbling_eng = events \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(window(col("event_time"), "5 minutes")) \
    .agg(_sum("engagement_score").alias("engagement"))

# Sliding: 10-min window, 2-min slide
sliding_eng = events \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(window(col("event_time"), "10 minutes", "2 minutes")) \
    .agg(_sum("engagement_score").alias("engagement"))

# Session: per-user sessions
session_eng = events \
    .withWatermark("event_time", "30 minutes") \
    .groupBy(
        session_window(col("event_time"), "15 minutes"),
        col("user_id")
    ) \
    .agg(_sum("engagement_score").alias("engagement"))




In [ ]:
+----------------------+------------------------+---------------------------+--------------------------+
| Metric               | Tumbling (5-min)       | Sliding (10-min/2-min)    | Session (15-min gap)     |
+----------------------+------------------------+---------------------------+--------------------------+
| Total engagement     | 25                      | 45 (overlapping)          | 14 (per user total)      |
| per window           |                        |                           |                          |
+----------------------+------------------------+---------------------------+--------------------------+
| Windows per hour     | 12                      | 30                        | Variable                 |
+----------------------+------------------------+---------------------------+--------------------------+
| Update frequency     | Every 5 min             | Every 2 min               | On session change        |
+----------------------+------------------------+---------------------------+--------------------------+
| Best for             | Periodic reporting      | Trend detection           | User behavior analysis   |
+----------------------+------------------------+---------------------------+--------------------------+
| State memory         | Low                     | Medium                    | High                     |
+----------------------+------------------------+---------------------------+--------------------------+
